In [ ]:
from pathlib import Path
from rdflib import ConjunctiveGraph

import sys
sys.path.append("../..")

path_data = Path("examples/data")

## Load (graph) data

In [ ]:
simulated_scenario_log_file = "example_1_1_event_log.json"
event_data_file = path_data.joinpath(simulated_scenario_log_file)

ekg_graph_rdf = ConjunctiveGraph()
ekg_graph_rdf.parse(event_data_file)

In [ ]:
from aggregated_traces.utils.construct_ekg import insert_DF_DP

ekg_graph = insert_DF_DP(ekg_graph_rdf)

## Insert fractions
**Note that this requires a complete trace graph!**<br>
Including all incoming and/or outgoing quantities for each lot in the path.

In [ ]:
# Check incoming vs outgoing amount
from aggregated_traces.utils.construct_ekg import check_quantities

query_result = check_quantities(ekg_graph)
print(query_result.serialize(format="txt").decode())

In [ ]:
# Insert fractions
from aggregated_traces.utils.construct_ekg import insert_fractions

ekg_graph = insert_fractions(ekg_graph)

## Create NetworkX graph

In [ ]:
from aggregated_traces.utils.construct_ekg import generateNetworkxDiGraph

ekg_graph_nx = generateNetworkxDiGraph(ekg_graph)

## Visualization

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

fig = generate_graph_visualization(ekg_graph_nx, f"output/{event_data_file.stem}")

# Traceability questions

In [ ]:
from pandas import set_option

set_option('display.max_colwidth', None)

### Backward
* What **equipment** was used to produce devices in packing unit X?
* With what propability was **equipment** used to produce an arbitrary device in packing unit X?

In [ ]:
# Define recources for which to retrieve the trace
entities_source_backward = [
    URIRef("http://example.org/id/ekg/aggregated_traces/Lot2_1_Pack0"),
    URIRef("http://example.org/id/ekg/aggregated_traces/Lot2_0_Pack0")
]

In [ ]:
from aggregated_traces.utils.ekg_analysis import compute_trace_probabilities

df_backward, edges_paths_backward = compute_trace_probabilities(rdf_trace_graph=ekg_graph, nx_trace_graph=ekg_graph_nx, source_entities=entities_source_backward, trace_backward=True)

df_backward["validated"] = df_backward["probability"] == df_backward["validation_fraction"]
if not df_backward["validated"].all():
    raise Exception("Not all probabilities match the validation fraction!")

df_backward.groupby(["entity_source", "entity_target"])[["probability"]].sum()

### Forward

* What packing units might be impacted by an issue on **equipment** (in a given time window)?
* With what probability did a device produced on **equipment** (in a given time window) end up in packing unit X?*

In [ ]:
entities_source_forward = [
    (URIRef("http://example.org/id/ekg/aggregated_traces/WB1"), Literal(10), Literal(15))
]

In [ ]:
df_forward, edges_paths_forward = compute_trace_probabilities(rdf_trace_graph=ekg_graph, nx_trace_graph=ekg_graph_nx, source_entities=entities_source_forward, trace_backward=False)

df_forward["validated"] = df_forward["probability"] == df_forward["validation_fraction"]
if not df_forward["validated"].all():
    raise Exception("Not all probabilities match the validation fraction!")

df_forward.groupby(["entity_source", "entity_target"])[["probability"]].sum()

### Visualization paths

In [ ]:
fig_paths = generate_graph_visualization(ekg_graph_nx, f"output/{event_data_file.stem}", edges_paths_backward, edges_paths_forward)